In [8]:
import os
import pandas as pd
import os

In [9]:
cur_dir = os.getcwd()

pypsa_data_dir = os.path.join(cur_dir, "../../Data/eq_handoff_v5/handoff")
expansion_scenario_list = ["conservative", "bau", "aggressive"]
agg_type = "disagg"

In [10]:
# Buses / substations
sites_df = None
for expansion_scenario in expansion_scenario_list:
    buses_all_fp = os.path.join(
        pypsa_data_dir,
        expansion_scenario,
        agg_type,
        "buses_all.csv"
    )
    buses_df = pd.read_csv(buses_all_fp)

    if sites_df is None:
        sites_df = buses_df[['lon', 'lat']].drop_duplicates().reset_index(drop=True)
        sites_df.insert(0, 'site_id', sites_df.index)
    else:
        # Find unique sites in buses_df that do not exist in sites_df
        new_sites = (
            buses_df[['lon', 'lat']]
            .drop_duplicates()
            .merge(sites_df[['lon', 'lat']], on=['lon', 'lat'],
                how='left', indicator=True)
            .query('_merge == "left_only"')[['lon', 'lat']]
            .reset_index(drop=True)
        )
        if not new_sites.empty:
            start_id = sites_df['site_id'].max() + 1
            new_sites.insert(0, 'site_id', range(start_id, start_id + len(new_sites)))
            sites_df = pd.concat([sites_df, new_sites], ignore_index=True)
    
    buses_df = buses_df.merge(sites_df, on=['lon', 'lat'], how='left')

    buses_all_output_fp = os.path.join(
            pypsa_data_dir,
            expansion_scenario,
            agg_type,
            "buses_all_aug.csv"
        )

    buses_df.to_csv(buses_all_output_fp, index=False)



In [11]:
print(len(sites_df))

8899


In [12]:
# Other assets
for expansion_scenario in expansion_scenario_list:
    assets_all_fp = os.path.join(
        pypsa_data_dir,
        expansion_scenario,
        agg_type,
        "assets_all.csv"
    )
    assets_df = pd.read_csv(assets_all_fp)
    # Find unique sites in assets_df that do not exist in sites_df
    new_sites = (
        assets_df[['lon', 'lat']]
        .drop_duplicates()
        .merge(sites_df[['lon', 'lat']], on=['lon', 'lat'],
            how='left', indicator=True)
        .query('_merge == "left_only"')[['lon', 'lat']]
        .reset_index(drop=True)
    )
    if not new_sites.empty:
        start_id = sites_df['site_id'].max() + 1
        new_sites.insert(0, 'site_id', range(start_id, start_id + len(new_sites)))
        sites_df = pd.concat([sites_df, new_sites], ignore_index=True)
    
    assets_df = assets_df.merge(sites_df, on=['lon', 'lat'], how='left')

    assets_all_output_fp = os.path.join(
            pypsa_data_dir,
            expansion_scenario,
            agg_type,
            "assets_all_aug.csv"
        )

    assets_df.to_csv(assets_all_output_fp, index=False)



In [13]:
sites_output_fp = os.path.join(
    pypsa_data_dir,
    "sites_all.csv"
)
print(len(sites_df))
sites_df.to_csv(sites_output_fp, index=False)

8899


In [14]:
assets_df.loc[100, ['lon', 'lat', 'site_id']]

lon       -118.324161
lat         37.142072
site_id           711
Name: 100, dtype: object

In [15]:
sites_df.loc[711,:]

site_id    711.000000
lon       -118.324161
lat         37.142072
Name: 711, dtype: float64